# 5.4 Modeling multi-body interactions

This notebook accompanies Section 5.4 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

#### This notebook provides an example of how to use the map equation to uncover modular structure in a hypergraph, by representing it as a bipartite, or multilayer network.


In [ ]:
!git clone https://github.com/mapequation/mapping-hypergraphs.git

In [ ]:
!cargo build --release --manifest-path mapping-hypergraphs/create-representations/Cargo.toml

Usage
```
cargo run -- representation hypergraph outfile
```
Where `representation` can be any of `-[b|B|u|U|m|M|hs|HS]` 

| Flag   | Network representation        |
|--------|-------------------------------|
| `-b`   | bipartite                     |
| `-B`   | bipartite non-lazy            |
| `-u`   | unipartite                    |
| `-U`   | unipartite non-lazy           |
| `-m`   | multilayer                    |
| `-M`   | multilayer non-lazy           |
| `-hs`  | hyperedge-similarity          |
| `-HS`  | hyperedge-similarity non-lazy |

#### Hypergraph Input Format

The hypergraph input file contains three sections:

1. **Vertices**  
   Each vertex has a numeric ID and an optional string label.

2. **Hyperedges**  
   Each hyperedge lists:
   - its ID  
   - the nodes it contains  
   - a hyperedge weight `omega`

3. **Weights (optional)**  
   Per-node hyperedge weights `gamma`.  
   If omitted, all node-hyperedge incidences default to **1**.

#### Example Hypergraph File

In [ ]:
# *Vertices
# # id [name]
# 1 "a"
# 2 "b"
# 3 "c"
# 4 "d"
# 5 "f"
# *Hyperedges
# # id nodes... omega
# 1 1 2 3 1
# 2 3 4 5 1
# *Weights
# # edge node gamma
# 1 1 1
# 1 2 1
# 1 3 2
# 2 3 2
# 2 4 1
# 2 5 1

In [ ]:
from infomap import Infomap
import networkx as nx
import subprocess
from pathlib import Path
import sys


In [ ]:
import _support as util


In [ ]:
def convert_hypergraph(input_filename, output_filename, representation_flag="-b"):
    """
    Convert a hypergraph file to a network representation using the Rust tool.

    Parameters
    ----------
    input_filename : str
        Path to the hypergraph file.
    output_filename : str
        Path for the output network file.
    representation_flag : str
        One of the supported network representation flags, default '-b' for bipartite.
    """
    input_path = Path(input_filename).resolve()
    output_path = Path(output_filename).resolve()

    cargo_manifest = Path(
        "mapping-hypergraphs/create-representations/Cargo.toml"
    ).resolve()

    cmd = [
        "cargo",
        "run",
        "--manifest-path",
        str(cargo_manifest),
        "--",
        representation_flag,
        str(input_path),
        str(output_path),
    ]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


# Example 1: Represent the input hypergraph as a bipartite network
convert_hypergraph(
    input_filename="mapping-hypergraphs/data/example-paper.txt",
    output_filename="output/example_bipartite.net",
    representation_flag="-b",
)

# Example 2: Represent the input hypergraph as a multilayer network
convert_hypergraph(
    input_filename="mapping-hypergraphs/data/example-paper.txt",
    output_filename="output/example_multilayer.net",
    representation_flag="-m",
)

In [ ]:
# Identify network modules using Infomap


def run_infomap(
    filename: str, columns=["node_id", "name", "flow", "module_id"], **infomap_args
):
    im = Infomap(silent=True, **infomap_args)
    im.read_file(filename)
    im.run()
    return im.get_dataframe(columns=columns)

In [ ]:
# Example 1: Bipartite network representation, where physical nodes and hyperedges are modeled as distinct node sets

filename_bipartite = "output/example_bipartite.net"
df_infomap_bipartite = run_infomap(
    filename_bipartite, skip_adjust_bipartite_flow=False, seed=123
).set_index("node_id")
df_infomap_bipartite


In [ ]:
# Example 2: Multilayer network representation, with each hyperedge forming a separate layer

filename_multilayer = "output/example_multilayer.net"
df_infomap_multilayer = run_infomap(
    filename_multilayer,
    ["state_id", "node_id", "layer_id", "name", "flow", "module_id"],
    seed=123,
).set_index("state_id")
df_infomap_multilayer


#### Visualize the hypergraph as a bipartite network. Nodes and hyperedges are colored according to their detected communities.

In [ ]:
sys.path.append("mapping-hypergraphs")
from hypergraph.network.hypergraph import HyperGraph


def parse_hypergraph(filename):
    with open(filename) as fp:
        H = HyperGraph.from_iter(fp)
        return H


H = parse_hypergraph("mapping-hypergraphs/data/example-paper.txt")

In [ ]:
def create_bipartite_graph(H: HyperGraph):
    weights = {(g.edge, g.node.id): g.gamma for g in H.weights}
    G = nx.Graph()
    N = len(H.nodes)
    for node in H.nodes:
        G.add_node(node.id, name=node.name, node_type="physical")
    for edge in H.edges:
        G.add_node(edge.id + N, name=edge.id, node_type="hyperedge")
        for n in edge.nodes:
            G.add_edge(edge.id + N, n.id, weight=weights[(edge.id, n.id)])
    return G


G = create_bipartite_graph(H)

In [ ]:
# Save Infomap modules as node attributes
for n in G:
    G.nodes[n]["module_id"] = df_infomap_bipartite.loc[n, "module_id"]

In [ ]:
util.plot_modular_network(
    G,
    node_color_attr="module_id",
    figsize=(4, 8),
    brighten_factor=0.3,
    use_value_as_palette_index=True,
    shape_attr="node_type",
    shape_map={"physical": "o", "hyperedge": "s"},
    show_labels=True,
    label_attr="name",
    bipartite_anchor_attr="node_type",
    bipartite_anchor_value="physical",
)